### RAG Pipelines- Data Ingestion to Vector DB Pipeline

In [1]:
import os
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

from llama_cloud_services import LlamaParse
from llama_index.core import Document
import re

from dotenv import load_dotenv
load_dotenv()

llama_api_key = os.getenv("LLAMA_CLOUD_API_KEY")

c:\Users\kirth\OneDrive\Desktop\AI_Software_Engineer_Kirthika\Github_Projects_for_AI\Agentic_AI_Overall\RAG_For_RACV_PolicyDocs\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\kirth\AppData\Local\Temp\ipykernel_14576\3916170459.py:5: DeprecationWarning: This package (llama-cloud-services) is deprecated and will be maintained until May 1, 2026. Please migrate to the new package: pip install llama-cloud>=1.0 (https://github.com/run-llama/llama-cloud-py). The new package provides the same functionality with improved performance and support.
  from llama_cloud_services import LlamaParse


In [2]:
import asyncio
import nest_asyncio
from llama_parse import LlamaParse
from llama_index.core import Document

# Patch the loop environment
nest_asyncio.apply()

async def process_all_pdfs_async():
    parsing_instructions = """
    This document is an insurance policy or roadside assistance terms and conditions document. 
    It contains benefit tables, eligibility conditions, coverage limits, and exclusion clauses. 
    Extract all tables as markdown tables preserving every row and column. 
    Preserve the relationship between coverage items and their corresponding limits. 
    Capture all specific quantities, distances, dollar amounts, and time limits exactly as written. 
    Do not merge separate clauses into single paragraphs.
    DO NOT ADD ANYTHING TO THE PDF DATA.
    """
    
    # Re-initialize the parser locally inside the running event loop context
    parser = LlamaParse(
        result_type="markdown",
        system_prompt=parsing_instructions,
        verbose=True
    )
    
    pdf_files = [
        "../data/pdfs/era-terms-and-conditions-2023-24.pdf",
        "../data/pdfs/motor-insurance-complete-care-pds-current.pdf"
    ]
    
    all_documents = []
    
    # Explicitly await files one by one to keep the async buffer clean
    for pdf_path in pdf_files:
        try:
            print(f"\nParsing via Cloud API: {pdf_path}")
            
            # Using aload_data keeps the background network loop active natively
            parsed_sections = await parser.aload_data(pdf_path)
            
            section_count = 0
            for doc in parsed_sections:
                clean_text = doc.text.replace('\t', ' ')
                clean_text = re.sub(r' +', ' ', clean_text)
                
                # Skip near-empty sections
                if len(clean_text.strip()) < 50:
                    continue
                
                all_documents.append(Document(
                    text=clean_text,
                    metadata={
                        "source_file": os.path.basename(pdf_path),
                        "file_type": "pdf"
                    }
                ))
                section_count += 1
                
            print(f" ✓ {os.path.basename(pdf_path)} — Loaded {section_count} cleaned chunks.")
            
        except Exception as e:
            print(f" ✗ Error parsing {pdf_path}: {e}")
            
    print(f"\nTotal clean chunks loaded globally: {len(all_documents)}")
    
    if all_documents:
        print("\nFirst true document preview:")
        print(all_documents[0].text[:500])
        
    return all_documents

# Trigger execution using the active asyncio loop framework
all_pdf_documents = asyncio.run(process_all_pdfs_async())


C:\Users\kirth\AppData\Local\Temp\ipykernel_14576\795833671.py:3: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse



Parsing via Cloud API: ../data/pdfs/era-terms-and-conditions-2023-24.pdf
Started parsing the file under job_id 9c30b94a-0017-4a11-aecd-c56bf8c6c577
 ✓ era-terms-and-conditions-2023-24.pdf — Loaded 16 cleaned chunks.

Parsing via Cloud API: ../data/pdfs/motor-insurance-complete-care-pds-current.pdf
Started parsing the file under job_id c9de9bda-af0f-4f20-975f-60fbd707b57c
 ✓ motor-insurance-complete-care-pds-current.pdf — Loaded 52 cleaned chunks.

Total clean chunks loaded globally: 68

First true document preview:
```markdown
| Coverage Item | Limit/Details |
|-----------------------------------|------------------------------------|
| Emergency Roadside Assistance | 24/7 service |
| Towing Distance | Up to 100 km |
| Battery Jump Start | Included |
| Flat Tyre Change | Included |
| Fuel Delivery | Up to 10 litres |
| Lockout Service | Included |
| Vehicle Breakdown | Included |
| Maximum Claim Limit | $1,500 per incident |
| Service Response Time | Within 60 minutes |
| Eligibility C

In [3]:
all_pdf_documents[:5]

[Document(id_='2db8c321-42ad-49c4-885c-0f73c9e06715', embedding=None, metadata={'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf'}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='```markdown\n| Coverage Item | Limit/Details |\n|-----------------------------------|------------------------------------|\n| Emergency Roadside Assistance | 24/7 service |\n| Towing Distance | Up to 100 km |\n| Battery Jump Start | Included |\n| Flat Tyre Change | Included |\n| Fuel Delivery | Up to 10 litres |\n| Lockout Service | Included |\n| Vehicle Breakdown | Included |\n| Maximum Claim Limit | $1,500 per incident |\n| Service Response Time | Within 60 minutes |\n| Eligibility Conditions | Must be a member |\n| Exclusions | Not applicable for unregistered vehicles |\n```\n', path=None, url=None, mimetype=None), image_resourc

In [4]:
from llama_index.core.node_parser import MarkdownNodeParser

def split_documents(documents):
    """
    Splits LlamaParse markdown documents into optimal RAG chunks (Nodes)
    while preserving markdown tables and structural headers intact.
    """
    # 1. Initialize the Markdown-aware parser
    # It automatically groups content logically by headers and markdown blocks
    markdown_parser = MarkdownNodeParser()
    
    print(f"Splitting {len(documents)} parsed document sections...")
    
    # 2. Get chunks (LlamaIndex calls these 'Nodes')
    split_nodes = markdown_parser.get_nodes_from_documents(documents)
    
    print(f"Successfully split into {len(split_nodes)} optimized RAG chunks.")
    
    # 3. Show a safe example chunk
    if split_nodes:
        print(f"\nExample chunk:")
        print(f"Content:\n{split_nodes[0].text[:300]}...")
        print(f"Metadata: {split_nodes[0].metadata}")
        
    return split_nodes

# Execute the chunking step
llama_chunks = split_documents(all_pdf_documents)


Splitting 68 parsed document sections...
Successfully split into 68 optimized RAG chunks.

Example chunk:
Content:
```markdown
| Coverage Item | Limit/Details |
|-----------------------------------|------------------------------------|
| Emergency Roadside Assistance | 24/7 service |
| Towing Distance | Up to 100 km |
| Battery Jump Start | Included |
| Flat Tyre Change | Included |
| Fuel Delivery | Up to 10 li...
Metadata: {'source_file': 'era-terms-and-conditions-2023-24.pdf', 'file_type': 'pdf', 'header_path': '/'}


In [5]:
type(llama_chunks)

list

### Embedding and VectorStoreDB

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "nomic-ai/nomic-embed-text-v1.5"):
        """Initialize embedding manager
        Args:
            model_name: HuggingFace model name for sentence embeddings"""

        self.model_name =  model_name
        self.model = None
        self._load_model()
    
    def _load_model(self):
        """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model =  SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimesions: {self.model.get_sentence_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name} : {e}")
            raise
    
    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """ Generate embeddings for a list of texts

            Args:
                texts: List of text strings to embed
            return:
                numpy array of embeddings with shape (len(texts), embedding_dim)
        """
        if not self.model:
            raise ValueError("Model not loaded")
        
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

## Initilize the Embedding manager
embedding_manager = EmbeddingManager()
embedding_manager

Loading embedding model: nomic-ai/nomic-embed-text-v1.5


c:\Users\kirth\OneDrive\Desktop\AI_Software_Engineer_Kirthika\Github_Projects_for_AI\Agentic_AI_Overall\RAG_For_RACV_PolicyDocs\.venv\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\kirth\.cache\huggingface\hub\models--nomic-ai--nomic-embed-text-v1.5. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Load

Model loaded successfully. Embedding dimesions: 768


C:\Users\kirth\AppData\Local\Temp\ipykernel_14576\3875187586.py:18: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print(f"Model loaded successfully. Embedding dimesions: {self.model.get_sentence_embedding_dimension()}")


### VectorStore

In [ ]:
class VectorStore:
    """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "../data/vector_store"):
        """ Initialize the vectore store

        Args:
            collection_name: Name of the ChromaDB collection
            persist_directory: Directory to persist the vector store
        """

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        """Initialize ChromDB client and collection"""
        try:
            os.makedirs(self.persist_directory, exist_ok = True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise 

    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        """
        Add documents and their embeddings to the vector store
        
        Args:
            documents: List of LangChain documents
            embeddings: Corresponding embeddings for the documents
        """
        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")
        
        print(f"Adding {len(documents)} documents to vector store...")
        
        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []
        
        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)
            
            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)
            
            # Document content
            documents_text.append(doc.page_content)
            
            # Embedding
            embeddings_list.append(embedding.tolist())
        
        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")
            
        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vectorstore=VectorStore()
vectorstore
    

In [ ]:
chunks

In [ ]:
texts=[doc.page_content for doc in chunks]
texts

In [ ]:
### Convert the text to embeddings
texts=[doc.page_content for doc in chunks]

## Generate the Embeddings

embeddings=embedding_manager.generate_embeddings(texts)

##store int he vector dtaabase
vectorstore.add_documents(chunks,embeddings)

### Retriever Pipeline From VectorStore

In [ ]:
class RAGRetriever:
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager):
        """
        Initialize the retriever

        Args:
            vector_store: Vector store containing document embeddings
            embedding_manager: Manager for generating query embeddings
        """

        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int=5, score_threshold: float= 0.0) -> List[Dict[str, Any]]:
        """
            Retrieve relevant documents for a query

            Args:
                query: The search query
                top_k: Number of top results to return
                score_threshold: Minimum similarity score threshold
            Returns:
                List of dictionaries containing retrieved documents and metadata
        """
        print(f"Retrieving documents for the query: {query}")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]

        try:
            results = self.vector_store.collection.query(
                query_embeddings = [query_embeddings.tolist()],
                n_results = top_k
            )

            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results ['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1-distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id' : doc_id,
                            'content' : document,
                            'metadata' : metadata,
                            'similarity_score' : similarity_score,
                            'distance' : distance,
                            'rank' : i+1
                        })
                    
                print(f"Retrieved {len(retrieved_docs)} docs (after filtering)")
                return retrieved_docs
                
            else:
                print(f"Error during retrieval: {e}")
                return []
        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []

rag_retriever = RAGRetriever(vectorstore, embedding_manager)                   

In [ ]:
rag_retriever

In [ ]:
rag_retriever.retrieve("Does RACV cover for accidental damage to your vehicle in Complete care?")

In [ ]:
rag_retriever.retrieve("How many litres of petrol does RACV provide in an emergency?")


In [ ]:
rag_retriever.retrieve("Tell me about Claiming under my Policy ?")


### Integration Vectordb Context pipeline With LLM output

In [ ]:
### Simple RAG pipeline with Groq LLM
from langchain_anthropic import ChatAnthropic
import os
from dotenv import load_dotenv
load_dotenv()

claude_api_key = os.getenv("CLAUDE_API_KEY")

llm = ChatAnthropic(anthropic_api_key=claude_api_key, model="claude-sonnet-4-5-20250929", temperature=0.1, max_tokens=1024)

## 2. Simple RAG function: retrieve context + generate response
def rag_simple(query, retriever, llm, top_k=3):
    ## retriever the context
    results=retriever.retrieve(query,top_k=top_k)

    context="\n\n".join([doc['content'] for doc in results]) if results else ""

    if not context:
        return "No relevant context found to answer the question."
    
    ## generate the answwer using GROQ LLM
    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:"""
    
    response=llm.invoke([prompt])
    return response.content

In [ ]:
answer = rag_simple("How many litres of petrol does RACV provide in an emergency?", rag_retriever, llm)
print(answer)

### Enhanced RAG Pipeline Features

In [ ]:
# --- Enhanced RAG Pipeline Features ---
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):
    """
    RAG pipeline with extra features:
    - Returns answer, sources, confidence score, and optionally full context.
    """
    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence': 0.0, 'context': ''}
    
    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
        'page': doc['metadata'].get('page', 'unknown'),
        'score': doc['similarity_score'],
        'preview': doc['content'][:300] + '...'
    } for doc in results]
    confidence = max([doc['similarity_score'] for doc in results])
    
    # Generate answer
    prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {query}\n\nAnswer:"""
    response = llm.invoke([prompt])
    
    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }
    if return_context:
        output['context'] = context
    return output

# Example usage:
result = rag_advanced("How many litres of petrol does RACV provide in an emergency?", rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])

In [ ]:
# --- Advanced RAG Pipeline: Streaming, Citations, History, Summarization ---
from typing import List, Dict, Any
import time

class AdvancedRAGPipeline:
    def __init__(self, retriever, llm):
        self.retriever = retriever
        self.llm = llm
        self.history = []  # Store query history

    def query(self, question: str, top_k: int = 5, min_score: float = 0.2, stream: bool = False, summarize: bool = False) -> Dict[str, Any]:
        # Retrieve relevant documents
        results = self.retriever.retrieve(question, top_k=top_k, score_threshold=min_score)
        if not results:
            answer = "No relevant context found."
            sources = []
            context = ""
        else:
            context = "\n\n".join([doc['content'] for doc in results])
            sources = [{
                'source': doc['metadata'].get('source_file', doc['metadata'].get('source', 'unknown')),
                'page': doc['metadata'].get('page', 'unknown'),
                'score': doc['similarity_score'],
                'preview': doc['content'][:120] + '...'
            } for doc in results]
            # Streaming answer simulation
            prompt = f"""Use the following context to answer the question concisely.\nContext:\n{context}\n\nQuestion: {question}\n\nAnswer:"""
            if stream:
                print("Streaming answer:")
                for i in range(0, len(prompt), 80):
                    print(prompt[i:i+80], end='', flush=True)
                    time.sleep(0.05)
                print()
            response = self.llm.invoke([prompt])
            answer = response.content

        # Add citations to answer
        citations = [f"[{i+1}] {src['source']} (page {src['page']})" for i, src in enumerate(sources)]
        answer_with_citations = answer + "\n\nCitations:\n" + "\n".join(citations) if citations else answer

        # Optionally summarize answer
        summary = None
        if summarize and answer:
            summary_prompt = f"Summarize the following answer in 2 sentences:\n{answer}"
            summary_resp = self.llm.invoke([summary_prompt])
            summary = summary_resp.content

        # Store query history
        self.history.append({
            'question': question,
            'answer': answer,
            'sources': sources,
            'summary': summary
        })

        return {
            'question': question,
            'answer': answer_with_citations,
            'sources': sources,
            'summary': summary,
            'history': self.history
        }

# Example usage:
adv_rag = AdvancedRAGPipeline(rag_retriever, llm)
result = adv_rag.query("what is attention is all you need", top_k=3, min_score=0.1, stream=True, summarize=True)
print("\nFinal Answer:", result['answer'])
print("Summary:", result['summary'])
print("History:", result['history'][-1])